# 📝 Day 5 Assignments — Auth: API Keys & JWT

Use `TestClient` so everything runs in Colab.


In [ ]:
!pip install fastapi uvicorn httpx python-jose[cryptography] passlib[bcrypt] python-multipart


## Task 1 — API Key Authentication

**Problem:** Build a dependency `verify_api_key(x_api_key: str = Header(...))` that:
- Looks up the key in `API_KEYS = {"secret-key-123": "alice"}`
- Returns the username on success
- Raises `HTTPException(401)` for any other key

Protect a `/whoami` endpoint with it.

**Expected output:**
```
No header                         → 422 (missing required header)
x-api-key=wrong                   → 401
x-api-key=secret-key-123          → {"user": "alice"}
```

💡 **Hint:** `from fastapi import Header, HTTPException, Depends`.


In [ ]:
from fastapi import FastAPI, Header, HTTPException, Depends
from fastapi.testclient import TestClient

API_KEYS = {"secret-key-123": "alice"}

# TODO: define verify_api_key dep

app = FastAPI()

# TODO: add protected /whoami endpoint

client = TestClient(app)
print(client.get("/whoami").status_code)
print(client.get("/whoami", headers={"x-api-key": "wrong"}).status_code)
print(client.get("/whoami", headers={"x-api-key": "secret-key-123"}).json())


## Task 2 — Bcrypt Password Hashing

**Problem:**
1. Hash the password `"correct-horse-battery-staple"` with bcrypt.
2. Verify the correct password returns `True`.
3. Verify a wrong password returns `False`.
4. Time how long `hash()` takes — bcrypt is intentionally slow.

**Expected output:**
```
hash: $2b$12$...
verify correct: True
verify wrong:   False
hash took ~0.2–0.5 seconds (good — that's the point)
```

💡 **Hint:** use `time.perf_counter()` around `pwd_context.hash(...)`.


In [ ]:
import time
from passlib.context import CryptContext

pwd_context = CryptContext(schemes=["bcrypt"], deprecated="auto")

# TODO: time the hash() call
# TODO: print verify for correct + wrong passwords


## Task 3 — Issuing & Decoding a JWT

**Problem:**
1. Encode a JWT with `sub="alice"` and 15-minute expiry. Print it.
2. Decode it and print the payload.
3. Encode a token with `exp` 1 second in the past. Confirm `jwt.decode` raises `JWTError`.

**Expected output:**
```
token: eyJ...
decoded: {"sub": "alice", "exp": ...}
expired -> ExpiredSignatureError
```

💡 **Hint:** use `datetime.now(timezone.utc) + timedelta(...)` for `exp`.


In [ ]:
from datetime import datetime, timedelta, timezone
from jose import jwt, JWTError

SECRET_KEY = "learn-me"
ALGORITHM = "HS256"

# TODO: encode a 15-min token, print it, decode it

# TODO: encode an already-expired token and confirm decode raises


## Task 4 — Full Login Flow

**Problem:** Build a FastAPI app with:
- An in-memory users dict where passwords are bcrypt-hashed
- `POST /login` that takes `OAuth2PasswordRequestForm`, verifies password, returns `{access_token, token_type}`
- `GET /me` protected by `get_current_user`, which decodes the JWT

Test with `TestClient`:
- `/login` with right password → 200, get a token
- `/login` with wrong password → 401
- `/me` without token → 401
- `/me` with token → returns the username

💡 **Hint:** `client.post("/login", data={"username": ..., "password": ...})` — `data=` (form), not `json=`.


In [ ]:
from datetime import datetime, timedelta, timezone
from fastapi import FastAPI, Depends, HTTPException, status
from fastapi.security import OAuth2PasswordBearer, OAuth2PasswordRequestForm
from fastapi.testclient import TestClient
from jose import jwt, JWTError
from passlib.context import CryptContext

pwd_context = CryptContext(schemes=["bcrypt"], deprecated="auto")
SECRET_KEY = "learn-me"
ALGORITHM = "HS256"

users = {"alice": {"username": "alice", "hashed_password": pwd_context.hash("secret")}}

app = FastAPI()
oauth2_scheme = OAuth2PasswordBearer(tokenUrl="login")

# TODO: define create_access_token(data, expires_delta)

# TODO: define get_current_user(token = Depends(oauth2_scheme))

# TODO: POST /login (uses OAuth2PasswordRequestForm)

# TODO: GET /me protected by get_current_user

client = TestClient(app)
print("bad pw:", client.post("/login", data={"username": "alice", "password": "nope"}).status_code)
r = client.post("/login", data={"username": "alice", "password": "secret"})
print("login:", r.json())
token = r.json().get("access_token")
print("/me no token:", client.get("/me").status_code)
print("/me with token:", client.get("/me", headers={"Authorization": f"Bearer {token}"}).json())


## 🎁 Bonus — `/refresh` Endpoint

**Problem:** Add a `POST /refresh` endpoint that:
- Takes a currently-valid token (via `Depends(oauth2_scheme)`)
- Decodes it and verifies the user exists
- Re-issues a new token with a fresh expiry (e.g. another 15 minutes)
- Returns `{access_token, token_type}`

💡 **Hint:** reuse `get_current_user` — if it returns successfully, the old token is valid; mint a new one with `create_access_token({"sub": user})`.


In [ ]:
# TODO: add @app.post("/refresh") that re-issues a token for the currently-authenticated user

# Then test with TestClient: get a token via /login, then call /refresh and compare the two tokens.


---

✅ When everything runs, you have a working bcrypt + JWT auth system. Next: rate limiting and OpenAPI docs.
